# Benchmark M1 trên Google Colab CPU-only

Owner: Hiếu (Member 3) · soạn hộ: Thiên, 27/10/2026. Soạn với sự hỗ trợ của AI assistant (Claude, Anthropic), ghi nguồn theo quy định môn học.

Chạy trên **runtime mới**: *Runtime → Change runtime type → CPU*, rồi **Run all**. Notebook đo đúng các số liệu đề yêu cầu:

- **Thời gian:** wall time end-to-end, thời gian discovery, thời gian trung bình mỗi bài (`/usr/bin/time -v`, `run_summary.json`).
- **Bộ nhớ:** RAM lớn nhất và trung bình là tổng RSS của mọi process (`src/utils/profiler.py`, lấy mẫu mỗi giây), disk lớn nhất, dung lượng output (`du`).

Thời gian cài đặt **không** tính vào wall time của crawler. Phạm vi benchmark (toàn dải hay mẫu) đang chờ TA xác nhận, nên đặt bằng `LIMIT`.

In [ ]:
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"
BRANCH = "main"
RANGE = "thien"  # dải dùng để benchmark
OFFSET = 0
LIMIT = 200  # None = cả dải 5.000 ID
WITH_REFERENCES = True  # đo cả Semantic Scholar chạy song song
OUT = "/content/bench"

In [ ]:
# Cấu hình máy: ghi vào report
!nproc
!lscpu | grep -E "Model name|^CPU\(s\)"
!free -h
!df -h /content
!python --version
!nvidia-smi -L || echo "Không có GPU: đúng runtime CPU-only"
!test -x /usr/bin/time || (apt-get -qq update && apt-get -qq install -y time)

In [ ]:
import os
import subprocess

if not os.path.isdir("/content/CodeRepo/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/CodeRepo"], check=True)
%cd /content/CodeRepo
!pip install -q -r src/requirements.txt
!git log --oneline -1

In [ ]:
from google.colab import userdata

for name in ("S2_API_KEY", "ARXIV_CONTACT_EMAIL"):
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: đã đặt")
    except Exception:  # secret chưa tạo hoặc chưa cấp quyền cho notebook
        print(f"{name}: CHƯA đặt trong Colab Secrets")

In [ ]:
SELECT = f"--range {RANGE} --offset {OFFSET}" + (f" --limit {LIMIT}" if LIMIT else "")
REFS = "" if WITH_REFERENCES else "--no-references"
PROFILER = (
    f"python -m src.utils.profiler --out {OUT}/profile"
    f" --watch {OUT}/data --watch {OUT}/work --output {OUT}/data"
)
PIPELINE = (
    f"python -m src.run_pipeline {SELECT} {REFS}"
    f" --data-root {OUT}/data/corpus --work-dir {OUT}/work --no-progress"
)
!rm -rf {OUT} && mkdir -p {OUT}/data {OUT}/work
!/usr/bin/time -v -o {OUT}/time_v.log {PROFILER} -- {PIPELINE}

In [ ]:
import json
import re
from pathlib import Path

prof = json.loads(Path(OUT, "profile", "profile_summary.json").read_text())
run = json.loads(Path(OUT, "work", "run_summary.json").read_text())
time_v = Path(OUT, "time_v.log").read_text()


def from_time_v(label):
    m = re.search(rf"{re.escape(label)}: (.+)", time_v)
    return m.group(1).strip() if m else None


report = {
    "wall time end-to-end (profiler, s)": prof["wall_s"],
    "wall time (/usr/bin/time)": from_time_v("Elapsed (wall clock) time (h:mm:ss or m:ss)"),
    "discovery (s)": run["discovery_s"],
    "trung bình mỗi bài (s)": run["avg_paper_s"],
    "RAM max, mọi process (MB)": prof["rss_max_mb"],
    "RAM trung bình (MB)": prof["rss_mean_mb"],
    "RAM max process chính (/usr/bin/time, kB)": from_time_v("Maximum resident set size (kbytes)"),
    "disk max (MB)": prof["disk_max_mb"],
    "output (MB)": prof["output_mb"],
    "bài thành công": f"{run['papers_complete']}/{run['ids_requested']}",
    "khoảng cách nhỏ nhất giữa 2 request arXiv (s)": run["min_request_gap_s"],
    "CPU / RAM máy": f"{prof['cpu_count']} vCPU, {prof['ram_total_mb']} MB",
}
for k, v in report.items():
    print(f"{k:<45} {v}")

In [ ]:
# CSV cho book (src/analysis/stats.py) và gói kết quả để tải về, đính kèm vào book repo
from google.colab import files

STATS = (
    f"python -m src.analysis.stats --data-root {OUT}/data/corpus --work-dir {OUT}/work"
    f" --profile-dir {OUT}/profile --time-log {OUT}/time_v.log --out {OUT}/book_csv"
)
KEEP = "time_v.log profile book_csv work/run_summary.json work/runs.jsonl work/run.log"
!{STATS}
!cd {OUT} && zip -qr bench_results.zip {KEEP}
files.download(f"{OUT}/bench_results.zip")